# Hướng dẫn sử dụng HSTrans với Encoder BRICS Motif 

Notebook này minh họa và so sánh hai chế độ mã hóa SMILES của thuốc cho bài toán dự đoán tác dụng phụ:
1. **BPE Subword Encoder (Gốc)**: Cắt chuỗi SMILES theo tần suất ký tự (vocab=2686, độ dài max=50).
2. **BRICS Motif Encoder (Mới)**: Cắt chuỗi SMILES thành các nhóm chức năng hóa học thực sự bằng thuật toán BRICS (vocab=799, độ dài max=20).


## 1. Môi trường và Import


In [3]:
import warnings
warnings.filterwarnings('ignore')
import torch
import pandas as pd
import numpy as np

# Import Subword (gốc)
from Net import drug2emb_encoder as bpe_encode

# Import Motif (mới)
from Net_BRICS import drug2motif_encoder as brics_encode
from Net_BRICS import compare_encoders, Trans_BRICS

ModuleNotFoundError: No module named 'Net'

## 2. So sánh cách mã hóa SMILES

Chúng ta sẽ xem cách mã hóa của 2 phương pháp với một số loại thuốc phổ biến.

In [ ]:
# Gọi hàm so sánh trực quan đã được viết trong Net_BRICS
compare_encoders()

## 3. Thử nghiệm Forward Pass của Model BRICS Mới

Khởi tạo model `Trans_BRIC` và truyền dữ liệu ảo (dummy data) để kiểm tra luồng hoạt động.

In [ ]:
model = Trans_BRICS().to('cpu')
print(f"Tổng số tham số: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

# Khởi tạo dữ liệu mẫu: Batch = 2
drug_ids = torch.randint(0, 799, (2, 20))       # [B, 20]
se_ids = torch.randint(0, 2686, (2, 50))       # [B, 50]
drug_mask = torch.ones(2, 20, dtype=torch.long) # [B, 20]
se_mask = torch.ones(2, 50, dtype=torch.long)   # [B, 50]

# Chạy qua model
score, _, _ = model(drug_ids, se_ids, drug_mask, se_mask)

print(f"\nInput Drug shape: {drug_ids.shape}")
print(f"Input Side Effect shape: {se_ids.shape}")
print(f"Output Score shape: {score.shape}")
print("Sample Output:", score.detach().flatten().numpy())

## 4. Hướng dẫn chạy Huấn luyện (Training)

Để tích hợp hoàn toàn mô hình BRICS vào vòng lặp huấn luyện, bạn cần điều chỉnh trong file `main.py`:

1. **Mở file `main.py`**
2. **Tìm dòng import:**
   ```python
   from Net import Trans, drug2emb_encoder
   ```
3. **Thay bằng:**
   ```python
   from Net_BRICS import Trans_BRICS as Trans
   from Net_BRICS import drug2motif_encoder as drug2emb_encoder
   ```
4. Các thông số input layer (ví dụ: `d_v, input_mask_d = drug2emb_encoder(d)`) có thể giữ nguyên vì API đã được thiết kế Drop-in thay thế hoàn hảo.
5. Sau đó, chạy lại lện `python main.py` hoặc thử chạy 1 epoch thông qua code dưới đây.

In [ ]:
# Code giả lập để chạy main.py từ bên trong notebook,
# Lưu ý bạn phải sửa file main.py theo bước 4 trước khi lệnh này có hiệu lực bằng BRICS.

import os
# !python main.py